In [227]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [228]:
data = pd.read_csv("Data/USO/USO_clean.csv")

**Statistics**

Time gap from last candle

In [229]:
data["timestamp"] = pd.to_datetime(data["timestamp"])
data["time_gap"] = data["timestamp"].diff().dt.total_seconds()/3600

Change hour to hour

In [230]:
data["internal_return"] = (data["close"] - data["open"])/ data["open"]

data["cc_return"] = data["close"].pct_change()

Volume

In [231]:
data["hour"] = pd.to_datetime(data["timestamp"]).dt.hour

data["hourly_typical_volume"] = (
    data.groupby("hour")["volume"]
      .transform(lambda s: s.shift(1).rolling(15, min_periods=3).mean())
)

data["relative_volume"] = (data["volume"]/data["hourly_typical_volume"].replace(0, np.nan))


Volatility

In [232]:
window = 10
min_periods = 5
data["volatility"] = data["cc_return"].rolling(window, min_periods=min_periods).std()

Target

In [233]:
next_close = data["close"].shift(-1)

data["target"] = np.nan
valid = next_close.notna()

data.loc[valid, "target"] = (
    next_close[valid] > data.loc[valid, "close"]
).astype(int)

In [234]:
data = data.drop(columns = ["timestamp", "open", "high", "low", "close", "volume"])
data.to_csv("Data/USO/USO_stats.csv")